# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution: Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process a dataset described by a Croissant schema using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`. 

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)
# Accessing the metadata.
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review available record sets, their fields, columns, and their `@id`s.

We'll list the record sets and their key details. All references will use the entities' `@id` fields, following best Croissant and `mlcroissant` practices.

In [ ]:
# List all record sets with their @id, name, and field/column @id's
record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets defined in the Croissant schema.")
else:
    for rs in record_sets:
        print(f"Record Set: {rs['@id']}")
        if hasattr(rs, 'name') or 'name' in rs:
            print(f"  Name: {getattr(rs, 'name', rs.get('name', ''))}")
        if hasattr(rs, 'field') or 'field' in rs:
            fields = getattr(rs, 'field', rs.get('field', []))
            if not isinstance(fields, list):
                fields = [fields]
            print("  Fields:")
            for f in fields:
                if isinstance(f, dict) and '@id' in f:
                    print(f"    - {f['@id']}")
                else:
                    print(f"    - {f}")
        # List columns if present
        if hasattr(rs, 'column') or 'column' in rs:
            columns = getattr(rs, 'column', rs.get('column', []))
            if not isinstance(columns, list):
                columns = [columns]
            print("  Columns:")
            for c in columns:
                if isinstance(c, dict) and '@id' in c:
                    print(f"    - {c['@id']}")
                else:
                    print(f"    - {c}")
        print("")

Let's preview some sample records from the available record sets (referencing by `@id`).

If you know the record set's `@id`, you can use it to iterate through the dataset:

In [ ]:
# List at most 1-2 record sets and show sample records by their @id
record_set_ids = [rs['@id'] if isinstance(rs, dict) else rs['@id'] for rs in dataset.record_sets]
for rs_id in record_set_ids:
    print(f"\nSample records from record set @{rs_id}:")
    for i, record in enumerate(dataset.records(record_set=rs_id)):
        print(record)
        if i >= 2:
            break

## 3. Data Extraction

Load data from each record set into a pandas DataFrame for analysis. We use record set and field `@id`s from the overview above.

In [ ]:
# Gather all record set @id values
record_set_ids = [rs['@id'] if isinstance(rs, dict) else rs['@id'] for rs in dataset.record_sets]

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading data for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        # dynamically create DataFrame if records exist
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"  Columns: {dataframes[record_set_id].columns.tolist()}")
        print(f"  Number of records: {len(dataframes[record_set_id])}")
        print(dataframes[record_set_id].head(2))
    else:
        print(f"  No records found for this set.")

# For demonstration, pick the first non-empty record set for subsequent cells:
if dataframes:
    primary_rs_id = next(iter(dataframes.keys()))
    print(f"Primary record set for analysis will be: {primary_rs_id}")
else:
    primary_rs_id = None

## 4. Exploratory Data Analysis (EDA)

Now, we'll apply processing to numeric and categorical fields.
- We'll choose a numeric field (by its `@id`) and perform filtering and normalization.
- If available, a categorical field (by `@id`) will be used for grouping.

Replace `numeric_field_id` and `group_field_id` below by appropriate column names (which correspond to the `@id`s of the Field or Column objects in Croissant; see previous cells' output).

In [ ]:
# If you know the correct @id for a numeric field, use it here. We'll list columns for reference:
if primary_rs_id:
    df = dataframes[primary_rs_id]
    print("Columns in DataFrame:")
    print(df.columns.tolist())

    # For the sake of example, we'll try to choose commonly present numeric fields. Replace with your actual @id as needed.
    # e.g., 'age', 'diagnosis_interval', etc. in Croissant may have ids like 'age', or full URIs.
    # Let's try to infer one numeric field:
    numeric_field_id = None
    for c in df.columns:
        if df[c].dtype in [np.int64, np.float64] or any(word in c.lower() for word in ['age', 'interval', 'years', 'count']):
            numeric_field_id = c
            break
    if numeric_field_id is None:
        raise ValueError("No numeric field found. Please edit and supply a numeric field @id.")
    print(f"Using '{numeric_field_id}' as numeric field for analysis.")

    # Set group_field by searching for typical categorical fields, like 'sex', 'gender', 'anatomy', etc.
    group_field = None
    for c in df.columns:
        if any(word in c.lower() for word in ['sex', 'gender', 'msi', 'location', 'site', 'group', 'category']):
            group_field = c
            break
    if group_field:
        print(f"Using '{group_field}' as group/categorical field.")

    # Filtering: For demonstration with numeric data - set a threshold. Here we use median as an example.
    if numeric_field_id:
        threshold = df[numeric_field_id].median() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df[[numeric_field_id]].head())

        normalized_col = f"{numeric_field_id}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Added normalized column '{normalized_col}':")
        print(filtered_df[[numeric_field_id, normalized_col]].head())

        # Group analysis by the group_field if available
        if group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"Mean of {numeric_field_id} grouped by {group_field}:")
            print(grouped_df.head())

## 5. Visualization

Visualize distributions or relationships in the data using Matplotlib.

In [ ]:
if primary_rs_id and numeric_field_id:
    import seaborn as sns
    df = dataframes[primary_rs_id]

    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f'{numeric_field_id} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion

This notebook demonstrated how to load, examine, and perform basic exploratory data analysis on the FAIR^2 dataset describing clinicopathological and molecular characteristics of second primary colorectal cancer in survivors, using the `mlcroissant` library.

- Data was accessed and referenced by Croissant `@id` values to ensure consistency.
- We loaded metadata, inspected record sets, and imported records as pandas DataFrames.
- Basic filtering, normalization, and grouping operations illustrated how to begin analyzing such FAIR datasets programmatically.

Explore further by using additional field `@id`s and custom analysis to answer your scientific questions!